## ⚙️ Colab 환경 설정 (처음 한 번만 실행)

> 💡 **로컬 환경**에서 실행하는 경우 이 셀들을 건너뛰세요.

In [ ]:
# ============================================================
# Colab 전용 설정 셀 — 로컬 실행 시 이 셀 전체를 건너뛰세요
# ============================================================
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 1) 저장소 클론 (또는 최신 코드 pull)
    REPO_DIR = '/content/gcp-media-ai-lab'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone',
                         'https://github.com/thesun4sky/gcp-media-ai-lab.git',
                         REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], check=True)
    os.chdir(REPO_DIR)
    print('현재 경로:', os.getcwd())

    # 2) Colab 전용 패키지 설치
    print('GCP 패키지 설치 중... (1~2분 소요)')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '-r', 'setup/requirements_colab.txt'],
        check=True
    )
    print('패키지 설치 완료')

    # 3) GCP 인증
    from google.colab import auth
    auth.authenticate_user()
    print('GCP 인증 완료')

    # 4) 프로젝트 ID 입력 (billing 활성화된 프로젝트 필요)
    print()
    print('💡 billing이 활성화된 GCP 프로젝트 ID를 입력하세요.')
    print('   예) gen-lang-client-0318067486')
    PROJECT_ID_INPUT = input('GCP 프로젝트 ID: ').strip()

    # 5) 환경변수 설정 — GCP 클라이언트 라이브러리가 읽는 모든 키 설정
    #    GOOGLE_CLOUD_QUOTA_PROJECT 가 가장 중요: quota/billing 프로젝트 지정
    os.environ['GOOGLE_CLOUD_PROJECT']       = PROJECT_ID_INPUT
    os.environ['GCLOUD_PROJECT']             = PROJECT_ID_INPUT
    os.environ['GOOGLE_CLOUD_QUOTA_PROJECT'] = PROJECT_ID_INPUT
    print(f'환경변수 설정 완료: {PROJECT_ID_INPUT}')

    # 6) gcloud 프로젝트 설정 및 API 활성화
    subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID_INPUT])
    APIS = [
        'videointelligence.googleapis.com',
        'speech.googleapis.com',
        'translate.googleapis.com',
        'aiplatform.googleapis.com',
        'bigquery.googleapis.com',
        'storage.googleapis.com',
    ]
    print('필요한 API 활성화 중...')
    subprocess.run(['gcloud', 'services', 'enable'] + APIS + [f'--project={PROJECT_ID_INPUT}'])
    print('API 활성화 완료')

    # 7) config.yaml 생성
    os.makedirs('setup', exist_ok=True)
    with open('setup/config.yaml', 'w') as f:
        f.write(f'gcp:\n  project_id: {PROJECT_ID_INPUT}\n'
                f'  region: asia-northeast3\n'
                f'  bucket_name: {PROJECT_ID_INPUT}-mediaai-lab\n')

    # 8) 이미 로드된 src 모듈 제거 → 최신 코드로 재임포트 보장
    for mod in list(sys.modules.keys()):
        if mod.startswith('src.'):
            del sys.modules[mod]

    print(f'\n✅ 설정 완료! project_id = {PROJECT_ID_INPUT}')

else:
    PROJECT_ID_INPUT = None  # 로컬에서는 config.yaml에서 자동 로드
    print('로컬 환경 감지 — Colab 설정 건너뜀')


# GCP Media AI 실습 개요

## DAN25: Media AI로 창작되고 소비되는 네이버 동영상

이 노트북은 GCP Media AI 실습의 전체 개요를 제공합니다.
각 셀을 순서대로 실행하여 Media AI의 핵심 기능을 체험할 수 있습니다.

### 학습 내용
1. Video Intelligence API - 영상 자동 분석
2. Speech-to-Text API - 자막 자동 생성
3. Translation API - 다국어 자막
4. 하이라이트 자동 추출
5. 콘텐츠 추천 시스템
6. Gemini 생성형 AI

## 0. 환경 설정

In [ ]:
# 필요 패키지 설치
import subprocess
import sys

# 설치 명령 (이미 설치되어 있다면 건너뜀)
# pip install은 위 Colab 설정 셀에서 이미 처리됩니다
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '../setup/requirements.txt'])

print('패키지 임포트 중...')
import os
import json
from pathlib import Path

# 프로젝트 루트 경로 설정
# Colab에서는 os.chdir()로 이미 프로젝트 루트에 있으므로 '.'를 사용
import os
project_root = Path(os.getcwd()).resolve() if 'google.colab' in sys.modules else Path('..').resolve()
sys.path.insert(0, str(project_root))

from src.gcp_client import load_config, get_project_id

print('환경 설정 완료!')

In [ ]:
# 설정 로드
config = load_config()
PROJECT_ID = get_project_id(config)
BUCKET_NAME = config.get('gcp', {}).get('bucket_name', f'{PROJECT_ID}-mediaai-lab')

print(f'프로젝트 ID: {PROJECT_ID}')
print(f'버킷 이름: {BUCKET_NAME}')
print()

# YOUR_PROJECT_ID가 그대로인 경우 경고
if PROJECT_ID == 'YOUR_PROJECT_ID':
    print('⚠️  경고: setup/config.yaml에서 PROJECT_ID를 실제 GCP 프로젝트 ID로 변경하세요.')
else:
    print(f'✅ 프로젝트 설정 확인: {PROJECT_ID}')

## Lab 1: Video Intelligence API

Google Cloud Video Intelligence API를 사용하여 영상에서
장면 전환, 레이블, 객체, 텍스트를 자동으로 감지합니다.

In [ ]:
# 아래는 샘플 데이터입니다. 실제 API 결과를 보려면 Cell 11을 먼저 실행하세요.
# Cell 11 실행 후 이 셀을 건너뛰고 바로 장면/레이블/객체 상세 셀을 실행하면 됩니다.

# Lab 1 샘플 출력 결과 확인
sample_output_path = project_root / 'labs/lab01_video_intelligence/sample_output.json'

with open(sample_output_path, 'r', encoding='utf-8') as f:
    sample_data = json.load(f)

print('=== Video Intelligence API 샘플 출력 ===')
print(f'분석 영상: {sample_data["video_uri"]}')
print(f'분석 시간: {sample_data["analysis_time"]}')
print(f'소요 시간: {sample_data["elapsed_seconds"]}초')
print()

# 장면 전환
shots = sample_data.get('shot_changes', [])
print(f'🎬 장면 전환: {len(shots)}개 장면 감지')
for shot in shots[:3]:
    print(f'  장면 {shot["shot_index"]+1}: {shot["start_time"]}s ~ {shot["end_time"]}s ({shot["duration"]}초)')
print()

# 상위 레이블
labels = sample_data.get('labels', [])
unique_labels = {}
for label in labels:
    if label['description'] not in unique_labels:
        unique_labels[label['description']] = label['confidence']

print(f'🏷️  레이블 (상위 5개):')
for desc, conf in sorted(unique_labels.items(), key=lambda x: x[1], reverse=True)[:5]:
    bar = '█' * int(conf * 20)
    print(f'  {desc:20s} {conf:.1%} {bar}')

In [ ]:
# ── 장면 전환 상세 출력 ──────────────────────────────────────
shots = sample_data.get('shot_changes', [])

print(f'🎬 장면 전환 상세 ({len(shots)}개)')
print(f'{"순위":^4}  {"시작":>7}  {"종료":>7}  {"길이":>6}  바 차트')
print('─' * 55)

max_dur = max(s['duration'] for s in shots)
for s in shots:
    bar_len = int(s['duration'] / max_dur * 30)
    bar = '█' * bar_len + '░' * (30 - bar_len)
    print(
        f'  {s["shot_index"]+1:2d}   '
        f'{s["start_time"]:6.1f}s  '
        f'{s["end_time"]:6.1f}s  '
        f'{s["duration"]:5.1f}s  '
        f'{bar}'
    )

total = sum(s['duration'] for s in shots)
avg   = total / len(shots) if shots else 0
print(f'\n  평균 장면 길이: {avg:.1f}초   총 영상 길이: {total:.1f}초')


In [ ]:
# ── 레이블 상세 출력 ─────────────────────────────────────────
labels = sample_data.get('labels', [])

# 레이블별로 최고 confidence 값만 유지
best = {}
for lb in labels:
    desc = lb['description']
    if desc not in best or lb['confidence'] > best[desc]['confidence']:
        best[desc] = lb
sorted_labels = sorted(best.values(), key=lambda x: x['confidence'], reverse=True)

print(f'🏷️  레이블 상세 ({len(sorted_labels)}개)')
print(f'{"레이블":<22} {"신뢰도":>6}  {"타입":<14}  신뢰도 바 차트')
print('─' * 72)

TYPE_ICON = {'segment_level': '📹 segment', 'shot_level': '🎞️  shot  '}
for lb in sorted_labels:
    bar_len = int(lb['confidence'] * 30)
    bar = '█' * bar_len + '░' * (30 - bar_len)
    icon = TYPE_ICON.get(lb['type'], lb['type'])
    print(
        f'  {lb["description"]:<20}  '
        f'{lb["confidence"]:5.1%}  '
        f'{icon}  '
        f'{bar}'
    )


In [ ]:
# ── 객체 추적 상세 출력 ───────────────────────────────────────
objects = sample_data.get('objects', [])

print(f'📦 객체 추적 상세 ({len(objects)}개 트랙)')
print(f'{"객체":<16} {"신뢰도":>6}  {"등장 시간":^16}  {"길이":>5}  {"프레임":>6}')
print('─' * 65)

for obj in sorted(objects, key=lambda x: x['confidence'], reverse=True):
    duration = obj['end_time'] - obj['start_time']
    time_range = f"{obj['start_time']:.1f}s ~ {obj['end_time']:.1f}s"
    print(
        f'  {obj["description"]:<14}  '
        f'{obj["confidence"]:5.1%}  '
        f'{time_range:^16}  '
        f'{duration:4.1f}s  '
        f'{obj["total_frames"]:5}프'
    )
    # 바운딩 박스 첫 프레임 표시
    if obj.get('frames'):
        bb = obj['frames'][0]['bounding_box']
        print(
            f'    └ 첫 프레임 bbox: '
            f'left={bb["left"]:.2f} top={bb["top"]:.2f} '
            f'right={bb["right"]:.2f} bottom={bb["bottom"]:.2f}'
        )


In [ ]:
# ── 실제 Video Intelligence API 호출 ────────────────────────
# 실행하면 실제 API를 호출하고 sample_data 변수를 덮어씁니다.
# 이후 장면/레이블/객체 상세 셀을 다시 실행하면 실제 결과가 표시됩니다.

import time
import datetime
from google.cloud import videointelligence
from src.gcp_client import get_video_intelligence_client

VIDEO_URI = 'gs://cloud-samples-data/video/animals.mp4'  # 원하는 GCS URI로 변경 가능
FEATURES = [
    videointelligence.Feature.SHOT_CHANGE_DETECTION,
    videointelligence.Feature.LABEL_DETECTION,
    videointelligence.Feature.OBJECT_TRACKING,
]

print(f'분석 영상: {VIDEO_URI}')
print('분석 중... (1~3분 소요)')
t0 = time.time()

client = get_video_intelligence_client()
operation = client.annotate_video(request={
    'input_uri': VIDEO_URI,
    'features': FEATURES,
})
result = operation.result(timeout=300)
elapsed = round(time.time() - t0, 1)
print(f'완료! ({elapsed}초 소요)')

annotation = result.annotation_results[0]

# ── 결과를 sample_data 형식으로 변환 (아래 상세 셀에서 재사용)
shot_changes = []
for i, shot in enumerate(annotation.shot_annotations):
    s = shot.start_time_offset.total_seconds()
    e = shot.end_time_offset.total_seconds()
    shot_changes.append({
        'shot_index': i,
        'start_time': round(s, 1),
        'end_time':   round(e, 1),
        'duration':   round(e - s, 1),
    })

labels = []
for seg_label in annotation.segment_label_annotations:
    for seg in seg_label.segments:
        labels.append({
            'description': seg_label.entity.description,
            'confidence':  round(seg.confidence, 4),
            'type':        'segment_level',
            'start_time':  seg.segment.start_time_offset.total_seconds(),
            'end_time':    seg.segment.end_time_offset.total_seconds(),
        })
for shot_label in annotation.shot_label_annotations:
    for seg in shot_label.segments:
        labels.append({
            'description': shot_label.entity.description,
            'confidence':  round(seg.confidence, 4),
            'type':        'shot_level',
            'start_time':  seg.segment.start_time_offset.total_seconds(),
            'end_time':    seg.segment.end_time_offset.total_seconds(),
        })

objects = []
for idx, track in enumerate(annotation.object_annotations):
    frames = [
        {
            'time': round(f.time_offset.total_seconds(), 2),
            'bounding_box': {
                'left':   round(f.normalized_bounding_box.left,   3),
                'top':    round(f.normalized_bounding_box.top,    3),
                'right':  round(f.normalized_bounding_box.right,  3),
                'bottom': round(f.normalized_bounding_box.bottom, 3),
            }
        }
        for f in track.frames
    ]
    s = track.segment.start_time_offset.total_seconds()
    e = track.segment.end_time_offset.total_seconds()
    objects.append({
        'description':  track.entity.description,
        'confidence':   round(track.confidence, 4),
        'track_id':     idx + 1,
        'start_time':   round(s, 1),
        'end_time':     round(e, 1),
        'frames':       frames,
        'total_frames': len(frames),
    })

# sample_data 덮어쓰기 → 아래 셀들이 실제 결과를 출력
sample_data = {
    'video_uri':       VIDEO_URI,
    'analysis_time':  str(datetime.datetime.now()),
    'elapsed_seconds': elapsed,
    'shot_changes':   shot_changes,
    'labels':         labels,
    'objects':        objects,
}

print(f'\n✅ 실제 분석 결과 저장 완료')
print(f'   장면: {len(shot_changes)}개')
print(f'   레이블: {len(labels)}개')
print(f'   객체 트랙: {len(objects)}개')
print('\n→ 아래 장면/레이블/객체 상세 셀을 실행하세요.')


## Lab 2: Speech-to-Text API

영상 음성을 텍스트로 변환하여 SRT/VTT 자막 파일을 생성합니다.

In [ ]:
# 샘플 SRT 파일 읽기
from src.subtitle_utils import parse_srt

sample_srt_path = project_root / 'labs/lab02_speech_subtitle/sample.srt'
srt_content = sample_srt_path.read_text(encoding='utf-8')
entries = parse_srt(srt_content)

print('=== 샘플 자막 파일 (SRT) ===')
print(f'총 자막 항목: {len(entries)}개')
print(f'영상 시작: {entries[0].start_time:.1f}초')
print(f'영상 종료: {entries[-1].end_time:.1f}초')
print()
print('자막 미리보기 (첫 5개):')
for entry in entries[:5]:
    print(f'[{entry.index}] {entry.start_time:.1f}s~{entry.end_time:.1f}s: {entry.text}')

In [ ]:
# 자막 통계
from src.subtitle_utils import calculate_subtitle_stats

stats = calculate_subtitle_stats(entries)
print('=== 자막 통계 ===')
for key, value in stats.items():
    print(f'  {key}: {value}')

## Lab 3: Translation API

한국어 자막을 영어, 일본어, 중국어로 자동 번역합니다.

In [ ]:
# Translation API 데모 (실제 API 호출 없이 구조 확인)
print('=== 번역 파이프라인 구조 ===')
print()
print('입력: 한국어 SRT 자막 파일')
print('  ↓')
print('1. SRT 파싱 → 자막 항목 추출')
print('  ↓')
print('2. Translation API v3 (일괄 번역)')
print('  - 배치 크기: 100항목')
print('  - 타임스탬프 보존')
print('  ↓')
print('3. 번역된 SRT/VTT 파일 저장')
print()
print('지원 언어:')
languages = {
    'en': '영어',
    'ja': '일본어', 
    'zh-CN': '중국어 간체',
    'zh-TW': '중국어 번체',
}
for code, name in languages.items():
    print(f'  {code}: {name}')

In [ ]:
# 번역 API 호출 (GCP 설정 완료 후 실행)
# 주석을 해제하고 실행하세요

# from google.cloud import translate_v3 as translate
# 
# client = translate.TranslationServiceClient()
# parent = f'projects/{PROJECT_ID}/locations/global'
# 
# # 자막 텍스트 번역
# texts = [entry.text for entry in entries[:3]]
# 
# response = client.translate_text(
#     request={
#         'parent': parent,
#         'contents': texts,
#         'source_language_code': 'ko',
#         'target_language_code': 'en',
#         'mime_type': 'text/plain',
#     }
# )
# 
# for i, translation in enumerate(response.translations):
#     print(f'원문: {texts[i]}')
#     print(f'번역: {translation.translated_text}')
#     print()
print('실제 API 호출은 주석을 해제하고 GCP 설정 후 실행하세요.')

## Lab 4: 하이라이트 자동 추출

영상 분석 결과를 기반으로 하이라이트 점수를 계산하고
최적의 장면을 자동으로 선택합니다.

In [ ]:
# 하이라이트 점수 계산 시뮬레이션
import random

# 샘플 장면 데이터 생성
random.seed(42)
sample_shots = []
for i in range(10):
    start = i * 30.0
    end = start + random.uniform(20, 40)
    
    # 가중치별 점수 계산
    label_score = random.uniform(0, 1) * 0.4
    object_score = random.uniform(0, 1) * 0.3
    text_score = random.choice([0, 0.1])
    diversity_score = random.uniform(0.3, 0.5) * 0.2
    total = label_score + object_score + text_score + diversity_score
    
    sample_shots.append({
        'index': i,
        'start': round(start, 1),
        'end': round(end, 1),
        'score': round(total, 4),
        'label_score': round(label_score, 4),
        'object_score': round(object_score, 4),
        'text_score': round(text_score, 4),
        'diversity_score': round(diversity_score, 4),
    })

# 점수 기준 정렬
sorted_shots = sorted(sample_shots, key=lambda x: x['score'], reverse=True)

print('=== 하이라이트 점수 랭킹 ===')
print(f'{'순위':4s} {'장면':8s} {'시간':20s} {'총점':8s} {'레이블':8s} {'객체':8s}')
print('-' * 60)
for rank, shot in enumerate(sorted_shots, 1):
    print(
        f'{rank:4d} '
        f'{shot["index"]+1:8d} '
        f'{shot["start"]:6.1f}s~{shot["end"]:6.1f}s '
        f'{shot["score"]:8.4f} '
        f'{shot["label_score"]:8.4f} '
        f'{shot["object_score"]:8.4f}'
    )

print(f'\n상위 3개 하이라이트 선택됨')

## Lab 5: 콘텐츠 추천 시스템

시청 이력 데이터를 기반으로 개인화 추천을 생성합니다.

In [ ]:
# 간단한 협업 필터링 시뮬레이션
import pandas as pd
import numpy as np

# 샘플 시청 이력 데이터
np.random.seed(42)

users = [f'user_{i:04d}' for i in range(1, 6)]
videos = [f'video_{i:04d}' for i in range(1, 11)]
categories = ['엔터테인먼트', '음악', '스포츠', '교육', '요리']

# 시청 완료율 행렬 (None = 시청 안 함)
watch_data = {
    'user_0001': [0.9, 0.8, None, None, 0.7, 0.6, None, 0.9, None, 0.8],
    'user_0002': [None, 0.7, 0.9, 0.8, None, None, 0.6, None, 0.9, None],
    'user_0003': [0.6, None, None, 0.9, 0.8, 0.7, None, 0.8, None, None],
    'user_0004': [None, None, 0.8, None, 0.9, None, 0.7, None, 0.6, 0.9],
    'user_0005': [0.7, 0.6, None, 0.8, None, 0.9, None, None, 0.8, None],
}

df = pd.DataFrame(watch_data, index=videos).T
print('=== 시청 이력 행렬 (완료율) ===')
print('None = 미시청')
print(df.to_string())
print()

# 간단한 코사인 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity

# NaN을 0으로 채워서 유사도 계산
matrix = df.fillna(0).values
similarity = cosine_similarity(matrix)
sim_df = pd.DataFrame(similarity, index=users, columns=users)

print('=== 사용자 간 유사도 행렬 ===')
print(sim_df.round(3).to_string())
print()

# user_0001에게 가장 유사한 사용자 찾기
target_user = 'user_0001'
similarities = sim_df[target_user].drop(target_user).sort_values(ascending=False)
most_similar_user = similarities.index[0]
print(f'{target_user}에게 가장 유사한 사용자: {most_similar_user} (유사도: {similarities[most_similar_user]:.3f})')

# 유사 사용자가 본 영상 중 target_user가 안 본 것 추천
target_watched = set(df.loc[target_user].dropna().index)
similar_watched = set(df.loc[most_similar_user].dropna().index)
recommendations = similar_watched - target_watched

print(f'\n추천 영상 (협업 필터링): {list(recommendations)}')

## Lab 6: 생성형 AI - Gemini

Gemini 1.5 Pro를 사용하여 영상 관련 콘텐츠를 자동 생성합니다.

In [ ]:
# 자막 텍스트 로드
sample_srt_path = project_root / 'labs/lab02_speech_subtitle/sample.srt'
srt_content = sample_srt_path.read_text(encoding='utf-8')
entries = parse_srt(srt_content)
full_text = ' '.join(entry.text for entry in entries)

print('=== 자막 텍스트 (처음 300자) ===')
print(full_text[:300])
print(f'...\n(총 {len(full_text)}자)')

In [ ]:
# Gemini API 호출 (GCP 설정 완료 후 실행)
# 주석을 해제하고 실행하세요

# import vertexai
# from vertexai.generative_models import GenerativeModel, GenerationConfig
# 
# vertexai.init(project=PROJECT_ID, location='asia-northeast3')
# model = GenerativeModel('gemini-1.5-flash')
# 
# prompt = f"""다음 영상 자막을 분석하여 영상 요약(3문장)과 태그 10개를 생성하세요.
# 
# 자막:
# {full_text[:1500]}
# 
# 응답 형식:
# 요약: [요약 내용]
# 태그: [#태그1, #태그2, ...]
# """
# 
# response = model.generate_content(
#     prompt,
#     generation_config=GenerationConfig(temperature=0.7, max_output_tokens=1024)
# )
# 
# print('=== Gemini 생성 결과 ===')
# print(response.text)
print('실제 API 호출은 주석을 해제하고 GCP 설정 후 실행하세요.')
print()
print('=== Gemini 예상 출력 예시 ===')
print()
print('요약:')
print('이 영상은 Google Cloud Platform의 Media AI 기술을 소개합니다. Video Intelligence API, Speech-to-Text API, Translation API, Vertex AI Gemini 등 다양한 GCP 서비스를 활용하여 영상 분석, 자막 생성, 다국어 지원, 하이라이트 추출, 콘텐츠 추천 기능을 구현하는 방법을 다룹니다. 네이버 동영상 플랫폼에서 AI가 창작과 소비 과정에 어떻게 활용되는지 실습을 통해 체험할 수 있습니다.')
print()
print('태그:')
print('#GCP #MediaAI #VideoIntelligence #SpeechToText #Translation #Gemini #VertexAI #자막자동생성 #AI영상분석 #네이버동영상')

## 전체 파이프라인 아키텍처

```
영상 업로드
     │
     ▼
┌─────────────────────────────────────────┐
│          Cloud Storage (GCS)            │
└─────────────────────────────────────────┘
     │
     ├──────────────────────────────────────┐
     │                                      │
     ▼                                      ▼
┌──────────────┐                    ┌──────────────┐
│ Video        │                    │ Speech-to-   │
│ Intelligence │                    │ Text API     │
│ API          │                    │              │
└──────────────┘                    └──────────────┘
     │                                      │
     │  장면/레이블/객체                    │  자막 텍스트
     │                                      │
     ▼                                      ▼
┌──────────────┐                    ┌──────────────┐
│ 하이라이트   │                    │ Translation  │
│ 추출         │                    │ API          │
└──────────────┘                    └──────────────┘
     │                                      │
     │  하이라이트 클립                     │  다국어 자막
     │                                      │
     └─────────────────┬────────────────────┘
                       │
                       ▼
                ┌──────────────┐
                │ Vertex AI    │
                │ Gemini 1.5   │
                │ Pro          │
                └──────────────┘
                       │
           ┌───────────┼───────────┐
           │           │           │
           ▼           ▼           ▼
        요약        태그        설명/제목
```


In [ ]:
# 실습 완료 체크리스트
print('=== 실습 완료 체크리스트 ===')
labs = [
    ('Lab 1', 'Video Intelligence API', 'labs/lab01_video_intelligence/analyze_video.py'),
    ('Lab 2', 'Speech-to-Text API (자막 생성)', 'labs/lab02_speech_subtitle/generate_subtitle.py'),
    ('Lab 3', 'Translation API (다국어 자막)', 'labs/lab03_multilingual_subtitle/translate_subtitle.py'),
    ('Lab 4', '하이라이트 자동 추출', 'labs/lab04_highlight_extraction/extract_highlights.py'),
    ('Lab 5', '콘텐츠 추천 시스템', 'labs/lab05_content_recommendation/recommendation_system.py'),
    ('Lab 6', 'Gemini 생성형 AI', 'labs/lab06_generative_ai/gemini_media_ai.py'),
]

for lab, desc, script in labs:
    script_path = project_root / script
    status = '✅ 파일 존재' if script_path.exists() else '❌ 파일 없음'
    print(f'  {status} [{lab}] {desc}')

print()
print('모든 파일이 준비되었습니다!')
print('GCP 설정 후 각 랩의 README.md를 따라 실습을 진행하세요.')